# Supervised Learning and Logistic Regression Lab

## Introduction

You are a junior data scientist working at a healthcare analytics company. You are assigned to develop a predictive model that helps doctors and hospitals assess a patient's risk of developing diabetes. With the increasing prevalence of diabetes, early prediction can significantly improve patient outcomes by enabling preventive care. The company has access to a medical dataset containing patient health metrics, including blood glucose levels, BMI, insulin levels, and age.

Currently, doctors use manual screening methods that rely on threshold-based risk assessments, which are not always accurate. The company wants to automate and enhance diabetes prediction using machine learning, specifically logistic regression, to classify whether a patient is at risk or not based on historical medical records.

The healthcare company needs a highly reliable diabetes prediction model to assist doctors with early diagnosis and preventive care. You must apply logistic regression to classify high-risk patients, ensuring that the model balances accuracy and interpretability while minimizing false negatives (missed diagnoses).

By completing this lab, you will gain valuable experience with data preprocessing, feature scaling, model tracking and evaluation, and performance metrics. This scenario mirrors real-world challenges in healthcare analytics, reinforcing key data science skills for medical risk assessment, patient health modeling, and AI-driven diagnostics.

### Step 0

In [1]:
# CodeGrade step0

# Import any useful libraries:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
# Run `pip install` only if the Faraway package is not installed:
#! pip install faraway

### Step 1: Load and Explore the Dataset

- Use the [Faraway](https://pypi.org/project/faraway/) package to load the Pima Indians Diabetes [dataset](https://rdrr.io/cran/faraway/man/pima.html), which contains patient health data.
- Convert categorical variables, in particular, the target variable `diabetes` indicator, to numerical values to ensure compatibility with the machine learning model.
- Display a preview of the dataset and check for missing values or inconsistencies.

In [3]:
# CodeGrade step1

# Import the dataset from Faraway:
from faraway.datasets import pima

# Load the dataset:
data = pima.load()

# Convert the `diabetes` column to an `int` type to represent classes 0 and 1.
# Remove the handful of 2s (unknown) in the data:
data["diabetes"] = data["diabetes"].astype(int)
data = data[data["diabetes"] != 2]

# Display the first few rows:
display(data.head())

# Display basic information:
data.info()

# Check for missing values:
display(data.isnull().sum())

# Return the shape of the data:
data.shape

,pregnant,glucose,diastolic,triceps,insulin,bmi,diabetes,age,test
0,6,148,72,35,0,33.6,0,50,1
1,1,85,66,29,0,26.6,0,31,0
2,8,183,64,0,0,23.3,0,32,1
3,1,89,66,23,94,28.1,0,21,0
5,5,116,74,0,0,25.6,0,30,0


<class 'pandas.core.frame.DataFrame'>
Index: 764 entries, 0 to 767
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   pregnant   764 non-null    int64  
 1   glucose    764 non-null    int64  
 2   diastolic  764 non-null    int64  
 3   triceps    764 non-null    int64  
 4   insulin    764 non-null    int64  
 5   bmi        764 non-null    float64
 6   diabetes   764 non-null    int64  
 7   age        764 non-null    int64  
 8   test       764 non-null    int64  
dtypes: float64(1), int64(8)
memory usage: 59.7 KB


pregnant     0
glucose      0
diastolic    0
triceps      0
insulin      0
bmi          0
diabetes     0
age          0
test         0
dtype: int64

(764, 9)

### Step 2: Preprocess the Data

- Identify relevant features (such as glucose level, BMI, insulin, and age) that may contribute to diabetes risk.
- Convert categorical variables if necessary and ensure that all numerical variables are in a consistent format.
- Prepare the feature data for scaling, but do not fit the scaler until after the training and testing sets have been created.
> Important: Avoid Data Leakage     
When scaling data, fit the scaler on the training data only. If you scale the full dataset before splitting, information from the test set is used to calculate the scaling values, such as the mean and standard deviation. This is called **data leakage**, and it can make the model appear to perform better than it would on truly unseen patient data. To avoid this, split the data first, then fit the scaler on `X_train` and use that same fitted scaler to transform both `X_train` and `X_test`.

In [4]:
# CodeGrade step2

# Define the features (X) and the target variable (y):
X = data[["glucose", "insulin", "bmi", "age"]]
y = data["diabetes"]

# Check the feature data types:
display(X.dtypes)

# Check the target classes:
display(y.value_counts())

glucose      int64
insulin      int64
bmi        float64
age          int64
dtype: object

diabetes
0    717
1     47
Name: count, dtype: int64

### Step 3: Split and Scale the Data

- Divide the dataset into **training** and **testing** sets to evaluate the model's performance.
- Use an appropriate ratio (e.g. 80% training, 20% testing) to balance model learning and evaluation.
- After splitting the data, standardize the numerical features. Fit the scaler on the training features only, then use that fitted scaler to transform both the training and testing features.

In [5]:
# CodeGrade step3

# Split the dataset into training (80%) and testing (20%) sets:
# The target variable, y, is imbalanced, with 717 zeros and only 47 ones. In
# order to keep this ratio approximately the same in our training and testing
# sets, we'll use stratified splitting.
seed = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, 
                                                    random_state = seed, stratify = y)

# Scale the features:
scaler = StandardScaler()

# Fit the scaler on the training data only:
X_train_scaled = scaler.fit_transform(X_train)

# Use this same fitted scaler to transform the testing data:
X_test_scaled = scaler.transform(X_test)

### Step 4: Train the Logistic Regression Model

- Apply a logistic regression classifier to the training dataset.
- Fit the model using the training data and ensure that it learns patterns in the data.
- Store the trained model for evaluation.

In [6]:
# CodeGrade step4

# Train the logistic regression model:
model = LogisticRegression().fit(X_train_scaled, y_train)

### Step 5: Make Predictions

- Use the trained model to predict diabetes risk for patients in the test set.
- Compare predicted labels against actual labels to assess model accuracy.

In [7]:
# CodeGrade step5

# Predict the probability of each patient in `X_test_scaled` having diabetes
# (class 1):
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# Using these probabilities, and the threshold that prob ≥ 0.5 --> 1/True,
# predict the diagnosis for each patient in `X_test_scaled`:
y_pred = model.predict(X_test_scaled)

# Print the first ten elements of both of these arrays:
print(f"First ten predicted probabilities: {y_pred_proba[0:10]}")
print(f"First ten predicted classes: {y_pred[0:10]}")

First ten predicted probabilities: [0.04842618 0.080447   0.04576153 0.05273184 0.06323505 0.05530943
 0.06217257 0.03372753 0.05005972 0.03434963]
First ten predicted classes: [0 0 0 0 0 0 0 0 0 0]


In [8]:
# Calculate the model's accuracy on the test set:
accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy:.4f}")

# Display the confusion matrix:
print(confusion_matrix(y_test, y_pred))

# Display the classification report:
print(classification_report(y_test, y_pred))

Test accuracy: 0.9412
[[144   0]
 [  9   0]]
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       144
           1       0.00      0.00      0.00         9

    accuracy                           0.94       153
   macro avg       0.47      0.50      0.48       153
weighted avg       0.89      0.94      0.91       153



/Users/briansilberman/miniforge3/envs/learn-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/briansilberman/miniforge3/envs/learn-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/briansilberman/miniforge3/envs/learn-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _